In [1]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('Basics').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 00:15:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [54]:
import os

In [7]:
files = os.listdir("../data/data/")[:10]

In [9]:
print(files)

['2130.json', '6870.json', '9510.json', '26310.json', '27690.json', '13770.json', '13320.json', '20010.json', '4230.json', '3330.json']


In [3]:
df = spark.read.option("multiline", "true").json("../data/data/")

In [4]:
df.count()

1050

In [5]:
df.printSchema()

root
 |-- collection: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- category: string (nullable = true)
 |    |    |-- license: string (nullable = true)
 |    |    |-- rel_abs: string (nullable = true)
 |    |    |-- rel_authors: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- author_inst: string (nullable = true)
 |    |    |    |    |-- author_name: string (nullable = true)
 |    |    |-- rel_date: string (nullable = true)
 |    |    |-- rel_doi: string (nullable = true)
 |    |    |-- rel_link: string (nullable = true)
 |    |    |-- rel_num_authors: long (nullable = true)
 |    |    |-- rel_site: string (nullable = true)
 |    |    |-- rel_title: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |    |    |-- version: string (nullable = true)
 |-- messages: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- count: long (nullabl

In [6]:
from pyspark.sql.functions import explode

In [15]:
df.show(1, True)

+--------------------+--------------------+
|          collection|            messages|
+--------------------+--------------------+
|[{public and glob...|[{30, 8340, ok, 3...|
+--------------------+--------------------+
only showing top 1 row


In [7]:
df_rows = df.select(explode("collection"))

In [8]:
df_rows.printSchema()

root
 |-- col: struct (nullable = true)
 |    |-- category: string (nullable = true)
 |    |-- license: string (nullable = true)
 |    |-- rel_abs: string (nullable = true)
 |    |-- rel_authors: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- author_inst: string (nullable = true)
 |    |    |    |-- author_name: string (nullable = true)
 |    |-- rel_date: string (nullable = true)
 |    |-- rel_doi: string (nullable = true)
 |    |-- rel_link: string (nullable = true)
 |    |-- rel_num_authors: long (nullable = true)
 |    |-- rel_site: string (nullable = true)
 |    |-- rel_title: string (nullable = true)
 |    |-- type: string (nullable = true)
 |    |-- version: string (nullable = true)



In [9]:
df_rows = df.select(explode("collection").alias("rel"))

In [10]:
df_rows.printSchema()

root
 |-- rel: struct (nullable = true)
 |    |-- category: string (nullable = true)
 |    |-- license: string (nullable = true)
 |    |-- rel_abs: string (nullable = true)
 |    |-- rel_authors: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- author_inst: string (nullable = true)
 |    |    |    |-- author_name: string (nullable = true)
 |    |-- rel_date: string (nullable = true)
 |    |-- rel_doi: string (nullable = true)
 |    |-- rel_link: string (nullable = true)
 |    |-- rel_num_authors: long (nullable = true)
 |    |-- rel_site: string (nullable = true)
 |    |-- rel_title: string (nullable = true)
 |    |-- type: string (nullable = true)
 |    |-- version: string (nullable = true)



In [11]:
df_rows_schema = df_rows.select("rel.*")

In [12]:
df_rows_schema.printSchema()

root
 |-- category: string (nullable = true)
 |-- license: string (nullable = true)
 |-- rel_abs: string (nullable = true)
 |-- rel_authors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- author_inst: string (nullable = true)
 |    |    |-- author_name: string (nullable = true)
 |-- rel_date: string (nullable = true)
 |-- rel_doi: string (nullable = true)
 |-- rel_link: string (nullable = true)
 |-- rel_num_authors: long (nullable = true)
 |-- rel_site: string (nullable = true)
 |-- rel_title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- version: string (nullable = true)



In [13]:
df_rows_schema.show(10)

+--------------------+-----------+--------------------+--------------------+----------+--------------------+--------------------+---------------+--------+--------------------+-------------------+-------+
|            category|    license|             rel_abs|         rel_authors|  rel_date|             rel_doi|            rel_link|rel_num_authors|rel_site|           rel_title|               type|version|
+--------------------+-----------+--------------------+--------------------+----------+--------------------+--------------------+---------------+--------+--------------------+-------------------+-------+
|public and global...|cc_by_nc_nd|Accurate predicti...|[{Mechanical Engi...|2022-04-21|10.1101/2022.04.2...|https://medrxiv.o...|              6| medRxiv|Accuracy of US CD...|PUBLISHAHEADOFPRINT|      1|
| infectious diseases|cc_by_nc_nd|IMPORTANCEMonoclo...|[{University of P...|2022-04-21|10.1101/2022.04.2...|https://medrxiv.o...|              8| medRxiv|Monoclonal Antibo...|PUBLISHAH

In [14]:
df_rows_schema.select("rel_doi", "rel_title").show(10)

+--------------------+--------------------+
|             rel_doi|           rel_title|
+--------------------+--------------------+
|10.1101/2022.04.2...|Accuracy of US CD...|
|10.1101/2022.04.2...|Monoclonal Antibo...|
|10.1101/2022.04.2...|Determinants of S...|
|10.1101/2022.04.2...|Elevated plasma C...|
|10.1101/2022.04.1...|A Platform for Da...|
|10.1101/2022.04.1...|A Statistical Arg...|
|10.1101/2022.04.2...|Genetic diversity...|
|10.1101/2022.04.1...|Immune response t...|
|10.1101/2022.04.1...|The impact of com...|
|10.1101/2022.04.1...|COVID-19 outcomes...|
+--------------------+--------------------+
only showing top 10 rows


In [15]:
df_rows_schema.select("rel_doi", "rel_title").show(10, False)

+---------------------------+------------------------------------------------------------------------------------------------------------------------------------------------+
|rel_doi                    |rel_title                                                                                                                                       |
+---------------------------+------------------------------------------------------------------------------------------------------------------------------------------------+
|10.1101/2022.04.20.22274097|Accuracy of US CDC COVID-19 Forecasting Models                                                                                                  |
|10.1101/2022.04.20.22274090|Monoclonal Antibodies for Treatment of SARS-CoV-2 Infection During Pregnancy                                                                    |
|10.1101/2022.04.21.22274025|Determinants of SARS-CoV-2 anti-spike antibody levels following BNT162b2 vaccination: cross-sect

In [16]:
df_author = df_rows_schema.select("rel_doi", "rel_authors")

In [17]:
df_author.show(1, False)

+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|rel_doi                    |rel_authors                                                                                                                                                                                                                                                          

In [18]:
df_author.printSchema()

root
 |-- rel_doi: string (nullable = true)
 |-- rel_authors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- author_inst: string (nullable = true)
 |    |    |-- author_name: string (nullable = true)



In [19]:
df_authors_tmp = df_author.withColumn("author", explode("rel_authors"))

In [20]:
df_authors_tmp.printSchema()

root
 |-- rel_doi: string (nullable = true)
 |-- rel_authors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- author_inst: string (nullable = true)
 |    |    |-- author_name: string (nullable = true)
 |-- author: struct (nullable = true)
 |    |-- author_inst: string (nullable = true)
 |    |-- author_name: string (nullable = true)



In [21]:
from pyspark.sql.functions import col

In [22]:
df_authors = df_authors_tmp.select(col("rel_doi").alias("doi"), col("author.author_name").alias("full_name"))

In [23]:
df_authors.printSchema()

root
 |-- doi: string (nullable = true)
 |-- full_name: string (nullable = true)



In [24]:
df_authors.show(10, False)

+---------------------------+-------------------+
|doi                        |full_name          |
+---------------------------+-------------------+
|10.1101/2022.04.20.22274097|Aviral Chharia     |
|10.1101/2022.04.20.22274097|Govind Jeevan      |
|10.1101/2022.04.20.22274097|Rajat Aayush Jha   |
|10.1101/2022.04.20.22274097|Meng Liu           |
|10.1101/2022.04.20.22274097|Jonathan M Berman  |
|10.1101/2022.04.20.22274097|Christin A Glorioso|
|10.1101/2022.04.20.22274090|Erin K McCreary    |
|10.1101/2022.04.20.22274090|Lara Lemon         |
|10.1101/2022.04.20.22274090|Christina Megli    |
|10.1101/2022.04.20.22274090|Amber Oakes        |
+---------------------------+-------------------+
only showing top 10 rows


In [25]:
df_authors.count()

346294

In [26]:
from pyspark.sql.functions import countDistinct, desc

In [27]:
df_author_count = df_authors.groupBy("full_name").agg(countDistinct("doi").alias("total_doi")).orderBy(desc("total_doi"))

In [28]:
df_author_count.show(20, False)

[Stage 15:========================================>               (24 + 8) / 33]

+--------------------+---------+
|full_name           |total_doi|
+--------------------+---------+
|Florian Krammer     |94       |
|Ben Goldacre        |74       |
|Amir Mehrkar        |74       |
|Rosalind M Eggo     |70       |
|Andrew Pekosz       |66       |
|Pei-Yong Shi        |65       |
|Brian MacKenna      |64       |
|Akiko Iwasaki       |63       |
|Wei Wang            |62       |
|David Veesler       |61       |
|Laith J Abu-Raddad  |60       |
|Alessandro Sette    |59       |
|Yang Liu            |58       |
|Christian Drosten   |58       |
|Adolfo Garcia-Sastre|56       |
|Roberto Bertollini  |55       |
|Hiam Chemaitelly    |55       |
|Jing Wang           |55       |
|Olivier Schwartz    |54       |
|Alex J Walker       |53       |
+--------------------+---------+
only showing top 20 rows


In [29]:
df_rows_schema.printSchema()

root
 |-- category: string (nullable = true)
 |-- license: string (nullable = true)
 |-- rel_abs: string (nullable = true)
 |-- rel_authors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- author_inst: string (nullable = true)
 |    |    |-- author_name: string (nullable = true)
 |-- rel_date: string (nullable = true)
 |-- rel_doi: string (nullable = true)
 |-- rel_link: string (nullable = true)
 |-- rel_num_authors: long (nullable = true)
 |-- rel_site: string (nullable = true)
 |-- rel_title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- version: string (nullable = true)



In [30]:
df_rows_schema.select("category", "rel_date", "rel_doi", "type", "version", "rel_title", "rel_link", "rel_site", "rel_abs").show(10)

+--------------------+----------+--------------------+-------------------+-------+--------------------+--------------------+--------+--------------------+
|            category|  rel_date|             rel_doi|               type|version|           rel_title|            rel_link|rel_site|             rel_abs|
+--------------------+----------+--------------------+-------------------+-------+--------------------+--------------------+--------+--------------------+
|public and global...|2022-04-21|10.1101/2022.04.2...|PUBLISHAHEADOFPRINT|      1|Accuracy of US CD...|https://medrxiv.o...| medRxiv|Accurate predicti...|
| infectious diseases|2022-04-21|10.1101/2022.04.2...|PUBLISHAHEADOFPRINT|      1|Monoclonal Antibo...|https://medrxiv.o...| medRxiv|IMPORTANCEMonoclo...|
| infectious diseases|2022-04-21|10.1101/2022.04.2...|PUBLISHAHEADOFPRINT|      1|Determinants of S...|https://medrxiv.o...| medRxiv|BackgroundUnderst...|
|cardiovascular me...|2022-04-21|10.1101/2022.04.2...|PUBLISHAHEADOFPR

In [31]:
df_raw = df_rows_schema.select("category", col("rel_date").alias("create_date"), col("rel_doi").alias("doi"), "type", "version", col("rel_title").alias("title"), col("rel_link").alias("link"), col("rel_site").alias("site"), col("rel_abs").alias("abstract"))

In [32]:
df_raw.printSchema()

root
 |-- category: string (nullable = true)
 |-- create_date: string (nullable = true)
 |-- doi: string (nullable = true)
 |-- type: string (nullable = true)
 |-- version: string (nullable = true)
 |-- title: string (nullable = true)
 |-- link: string (nullable = true)
 |-- site: string (nullable = true)
 |-- abstract: string (nullable = true)



In [33]:
df_raw.show(10)

+--------------------+-----------+--------------------+-------------------+-------+--------------------+--------------------+-------+--------------------+
|            category|create_date|                 doi|               type|version|               title|                link|   site|            abstract|
+--------------------+-----------+--------------------+-------------------+-------+--------------------+--------------------+-------+--------------------+
|public and global...| 2022-04-21|10.1101/2022.04.2...|PUBLISHAHEADOFPRINT|      1|Accuracy of US CD...|https://medrxiv.o...|medRxiv|Accurate predicti...|
| infectious diseases| 2022-04-21|10.1101/2022.04.2...|PUBLISHAHEADOFPRINT|      1|Monoclonal Antibo...|https://medrxiv.o...|medRxiv|IMPORTANCEMonoclo...|
| infectious diseases| 2022-04-21|10.1101/2022.04.2...|PUBLISHAHEADOFPRINT|      1|Determinants of S...|https://medrxiv.o...|medRxiv|BackgroundUnderst...|
|cardiovascular me...| 2022-04-21|10.1101/2022.04.2...|PUBLISHAHEADOFP

In [34]:
from pyspark.sql.functions import initcap, to_date, date_format

In [35]:
df_raw_clean = df_raw.withColumn("category", initcap("category")).withColumn("create_date", date_format( to_date("create_date", "yyyy-MM-dd"), "MM/dd/yyyy"))

In [36]:
df_raw_clean.printSchema()

root
 |-- category: string (nullable = true)
 |-- create_date: string (nullable = true)
 |-- doi: string (nullable = true)
 |-- type: string (nullable = true)
 |-- version: string (nullable = true)
 |-- title: string (nullable = true)
 |-- link: string (nullable = true)
 |-- site: string (nullable = true)
 |-- abstract: string (nullable = true)



In [37]:
df_raw_clean.show(10)

[Stage 23:>                                                         (0 + 1) / 1]

+--------------------+-----------+--------------------+-------------------+-------+--------------------+--------------------+-------+--------------------+
|            category|create_date|                 doi|               type|version|               title|                link|   site|            abstract|
+--------------------+-----------+--------------------+-------------------+-------+--------------------+--------------------+-------+--------------------+
|Public And Global...| 04/21/2022|10.1101/2022.04.2...|PUBLISHAHEADOFPRINT|      1|Accuracy of US CD...|https://medrxiv.o...|medRxiv|Accurate predicti...|
| Infectious Diseases| 04/21/2022|10.1101/2022.04.2...|PUBLISHAHEADOFPRINT|      1|Monoclonal Antibo...|https://medrxiv.o...|medRxiv|IMPORTANCEMonoclo...|
| Infectious Diseases| 04/21/2022|10.1101/2022.04.2...|PUBLISHAHEADOFPRINT|      1|Determinants of S...|https://medrxiv.o...|medRxiv|BackgroundUnderst...|
|Cardiovascular Me...| 04/21/2022|10.1101/2022.04.2...|PUBLISHAHEADOFP

In [38]:
df_raw_clean.count()

31476

In [70]:
df_raw_clean.write.mode("overwrite").json("../data/output/df_raw_clean1")

In [73]:
df.rdd.getNumPartitions()

33

In [74]:
df_raw_clean.coalesce(4).write.mode("overwrite").json("../data/output/df_raw_clean2")

In [75]:
df_raw_clean.repartition(100).write.mode("overwrite").json("../data/output/df_raw_clean3")

In [76]:
df_raw_clean.write.mode("overwrite").parquet("../data/output/df_raw_clean4")

26/05/10 17:57:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/10 17:57:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/10 17:57:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/10 17:57:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [77]:
df_raw_clean.repartition(100).write.mode("overwrite").parquet("../data/output/df_raw_clean5")

26/05/10 17:58:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [78]:
df_raw_clean.coalesce(4).write.mode("overwrite").parquet("../data/output/df_raw_clean6")

In [39]:
df_raw_clean.printSchema()

root
 |-- category: string (nullable = true)
 |-- create_date: string (nullable = true)
 |-- doi: string (nullable = true)
 |-- type: string (nullable = true)
 |-- version: string (nullable = true)
 |-- title: string (nullable = true)
 |-- link: string (nullable = true)
 |-- site: string (nullable = true)
 |-- abstract: string (nullable = true)



In [40]:
df_authors.printSchema()

root
 |-- doi: string (nullable = true)
 |-- full_name: string (nullable = true)



In [41]:
df_joined = df_raw_clean.join(df_authors, on="doi", how="inner")

In [42]:
df_joined.show(10)

[Stage 28:========================================>               (24 + 9) / 33]

+--------------------+--------------+-----------+-------------------+-------+--------------------+--------------------+-------+--------------------+-------------------+
|                 doi|      category|create_date|               type|version|               title|                link|   site|            abstract|          full_name|
+--------------------+--------------+-----------+-------------------+-------+--------------------+--------------------+-------+--------------------+-------------------+
|10.1101/2020.01.2...|Bioinformatics| 01/24/2020|        new results|      1|The 2019-new Coro...|https://biorxiv.o...|bioRxiv|There is concern ...| Domenico Benvenuto|
|10.1101/2020.01.2...|Bioinformatics| 01/24/2020|        new results|      1|The 2019-new Coro...|https://biorxiv.o...|bioRxiv|There is concern ...|   Marta Giovanetti|
|10.1101/2020.01.2...|Bioinformatics| 01/24/2020|        new results|      1|The 2019-new Coro...|https://biorxiv.o...|bioRxiv|There is concern ...|Alessan

In [43]:
df_joined.printSchema()

root
 |-- doi: string (nullable = true)
 |-- category: string (nullable = true)
 |-- create_date: string (nullable = true)
 |-- type: string (nullable = true)
 |-- version: string (nullable = true)
 |-- title: string (nullable = true)
 |-- link: string (nullable = true)
 |-- site: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- full_name: string (nullable = true)



In [44]:
df_joined.count()

346294

In [45]:
from pyspark.sql.functions import collect_list

In [46]:
df_grouped = df_joined.groupBy(
        "doi",
        "category",
        "create_date",
        "type",
        "version",
        "title",
        "link",
        "site",
        "abstract"
    ).agg(
        collect_list("full_name").alias("authors")
    )

In [47]:
df_grouped.show(10)

[Stage 36:=======================================>                (23 + 8) / 33]

+--------------------+-------------------+-----------+-------------------+-------+--------------------+--------------------+-------+--------------------+--------------------+
|                 doi|           category|create_date|               type|version|               title|                link|   site|            abstract|             authors|
+--------------------+-------------------+-----------+-------------------+-------+--------------------+--------------------+-------+--------------------+--------------------+
|10.1101/2020.01.2...|     Bioinformatics| 01/24/2020|        new results|      1|The 2019-new Coro...|https://biorxiv.o...|bioRxiv|There is concern ...|[Domenico Benvenu...|
|10.1101/2020.01.2...|       Epidemiology| 01/28/2020|PUBLISHAHEADOFPRINT|      1|Epidemiological i...|https://medrxiv.o...|medRxiv|ObjectiveVirologi...|[Sung-mok Jung, R...|
|10.1101/2020.01.2...|       Microbiology| 01/28/2020|        new results|      1|Breaking down of ...|https://biorxiv.o...|b

In [48]:
df_grouped.printSchema()

root
 |-- doi: string (nullable = true)
 |-- category: string (nullable = true)
 |-- create_date: string (nullable = true)
 |-- type: string (nullable = true)
 |-- version: string (nullable = true)
 |-- title: string (nullable = true)
 |-- link: string (nullable = true)
 |-- site: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- authors: array (nullable = false)
 |    |-- element: string (containsNull = false)



In [49]:
from pyspark.sql.functions import collect_set

In [50]:
df_grouped = df_joined.groupBy(
        "doi",
        "category",
        "create_date",
        "type",
        "version",
        "title",
        "link",
        "site",
        "abstract"
    ).agg(
        collect_set("full_name").alias("authors")
    )

In [51]:
df_grouped.show(10)

[Stage 42:========================================>               (24 + 8) / 33]

+--------------------+-------------------+-----------+-------------------+-------+--------------------+--------------------+-------+--------------------+--------------------+
|                 doi|           category|create_date|               type|version|               title|                link|   site|            abstract|             authors|
+--------------------+-------------------+-----------+-------------------+-------+--------------------+--------------------+-------+--------------------+--------------------+
|10.1101/2020.01.2...|     Bioinformatics| 01/24/2020|        new results|      1|The 2019-new Coro...|https://biorxiv.o...|bioRxiv|There is concern ...|[Silvia Spoto, Ma...|
|10.1101/2020.01.2...|       Epidemiology| 01/28/2020|PUBLISHAHEADOFPRINT|      1|Epidemiological i...|https://medrxiv.o...|medRxiv|ObjectiveVirologi...|[Robin N. Thompso...|
|10.1101/2020.01.2...|       Microbiology| 01/28/2020|        new results|      1|Breaking down of ...|https://biorxiv.o...|b

In [52]:
df_grouped.printSchema()

root
 |-- doi: string (nullable = true)
 |-- category: string (nullable = true)
 |-- create_date: string (nullable = true)
 |-- type: string (nullable = true)
 |-- version: string (nullable = true)
 |-- title: string (nullable = true)
 |-- link: string (nullable = true)
 |-- site: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- authors: array (nullable = false)
 |    |-- element: string (containsNull = false)



In [55]:
crossref = os.listdir("../data/crossref/")[:10]

In [97]:
print(crossref)

['10.1101_2021.03.24.21254124.json', '10.1101_2021.07.19.452910.json', '10.1101_2020.04.17.20069716.json', '10.1101_2020.04.11.20056523.json', '10.1101_2020.06.03.20119925.json', '10.1101_2022.03.02.22271779.json', '10.1101_2022.06.19.22276620.json', '10.1101_2022.12.06.519378.json', '10.1101_2021.12.28.21268435.json', '10.1101_2021.09.10.21263332.json']


In [56]:
cr = spark.read.option("multiline", "true").json("../data/crossref/")

In [57]:
cr.count()

26/05/10 19:39:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

31458

In [58]:
cr.printSchema()

root
 |-- message: struct (nullable = true)
 |    |-- DOI: string (nullable = true)
 |    |-- URL: string (nullable = true)
 |    |-- abstract: string (nullable = true)
 |    |-- accepted: struct (nullable = true)
 |    |    |-- date-parts: array (nullable = true)
 |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |-- element: long (containsNull = true)
 |    |-- author: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- ORCID: string (nullable = true)
 |    |    |    |-- affiliation: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- authenticated-orcid: boolean (nullable = true)
 |    |    |    |-- family: string (nullable = true)
 |    |    |    |-- given: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- sequence: string (nullable = true)
 |    |    

In [59]:
df_authors.show(10, False)

+---------------------------+-------------------+
|doi                        |full_name          |
+---------------------------+-------------------+
|10.1101/2022.04.20.22274097|Aviral Chharia     |
|10.1101/2022.04.20.22274097|Govind Jeevan      |
|10.1101/2022.04.20.22274097|Rajat Aayush Jha   |
|10.1101/2022.04.20.22274097|Meng Liu           |
|10.1101/2022.04.20.22274097|Jonathan M Berman  |
|10.1101/2022.04.20.22274097|Christin A Glorioso|
|10.1101/2022.04.20.22274090|Erin K McCreary    |
|10.1101/2022.04.20.22274090|Lara Lemon         |
|10.1101/2022.04.20.22274090|Christina Megli    |
|10.1101/2022.04.20.22274090|Amber Oakes        |
+---------------------------+-------------------+
only showing top 10 rows


In [60]:
df_authors.printSchema()

root
 |-- doi: string (nullable = true)
 |-- full_name: string (nullable = true)



In [61]:
from pyspark.sql.functions import lower

In [66]:
df_authors.filter(
    lower(col("full_name")).contains("yuanyuan")
).count()

48

In [67]:
df_authors.filter(
    lower(col("full_name")).contains("yuanyuan")
).show(48, False)

+---------------------------+---------------+
|doi                        |full_name      |
+---------------------------+---------------+
|10.1101/2022.05.27.493682  |Yuanyuan Qu    |
|10.1101/2024.03.04.583178  |Xinyuanyuan Sun|
|10.1101/2020.11.11.375972  |Yuanyuan Zhang |
|10.1101/2020.11.12.378422  |Xinyuanyuan Sun|
|10.1101/2020.10.08.20209114|Yuanyuan Zhang |
|10.1101/2020.08.07.20169847|Yuanyuan Qu    |
|10.1101/2021.02.03.21251089|Yuanyuan Shi   |
|10.1101/2022.09.25.509344  |Yuanyuan Zhang |
|10.1101/2021.12.22.473478  |Yuanyuan Li    |
|10.1101/2022.05.21.492903  |Yuanyuan Zhang |
|10.1101/2020.05.02.043554  |Xinyuanyuan Sun|
|10.1101/2021.04.16.440083  |Yuanyuan Qu    |
|10.1101/2025.01.10.632062  |Yuanyuan Zhang |
|10.1101/2022.07.31.502215  |Xinyuanyuan Sun|
|10.1101/2022.03.07.22272055|Yuanyuan Qiu   |
|10.1101/2021.08.04.21261561|Yuanyuan Shi   |
|10.1101/2020.08.31.274704  |Yuanyuan Zhang |
|10.1101/2020.11.12.380931  |Yuanyuan Yan   |
|10.1101/2020.03.02.20030064|Yuany

In [68]:
cr.printSchema()

root
 |-- message: struct (nullable = true)
 |    |-- DOI: string (nullable = true)
 |    |-- URL: string (nullable = true)
 |    |-- abstract: string (nullable = true)
 |    |-- accepted: struct (nullable = true)
 |    |    |-- date-parts: array (nullable = true)
 |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |-- element: long (containsNull = true)
 |    |-- author: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- ORCID: string (nullable = true)
 |    |    |    |-- affiliation: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- authenticated-orcid: boolean (nullable = true)
 |    |    |    |-- family: string (nullable = true)
 |    |    |    |-- given: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- sequence: string (nullable = true)
 |    |    

In [70]:
cr_rows = cr.select("message.*")

In [71]:
cr_rows.printSchema()

root
 |-- DOI: string (nullable = true)
 |-- URL: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- accepted: struct (nullable = true)
 |    |-- date-parts: array (nullable = true)
 |    |    |-- element: array (containsNull = true)
 |    |    |    |-- element: long (containsNull = true)
 |-- author: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- ORCID: string (nullable = true)
 |    |    |-- affiliation: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |-- authenticated-orcid: boolean (nullable = true)
 |    |    |-- family: string (nullable = true)
 |    |    |-- given: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- sequence: string (nullable = true)
 |    |    |-- suffix: string (nullable = true)
 |-- container-title: array (nullable = true)
 |    |-- element: string (containsNull = true

In [74]:
cr_author_tmp = cr_rows.select(col("DOI").alias("doi"), explode("author").alias("author")).select("doi", "author.*")

In [75]:
cr_author_tmp.printSchema()

root
 |-- doi: string (nullable = true)
 |-- ORCID: string (nullable = true)
 |-- affiliation: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |-- authenticated-orcid: boolean (nullable = true)
 |-- family: string (nullable = true)
 |-- given: string (nullable = true)
 |-- name: string (nullable = true)
 |-- sequence: string (nullable = true)
 |-- suffix: string (nullable = true)



In [76]:
cr_author_tmp.select("name","given", "family" ).show(100)

+----+-----------------+--------------------+
|name|            given|              family|
+----+-----------------+--------------------+
|NULL|   Christopher T.|             Leffler|
|NULL|            Edsel|                 Ing|
|NULL|        Joseph D.|              Lykins|
|NULL|       Matthew C.|               Hogan|
|NULL|         Craig A.|             McKeown|
|NULL|          Andrzej|          Grzybowski|
|NULL|       Emmanuelle|             Logette|
|NULL|        Charlotte|               Lorin|
|NULL|          Cyrille|             Favreau|
|NULL|          Eugenia|             Oshurko|
|NULL|           Jay S.|              Coggan|
|NULL|        Francesco|           Casalegno|
|NULL|Mohameth François|                  Sy|
|NULL|          Caitlin|              Monney|
|NULL|           Marine|            Bertschy|
|NULL|           Emilie|            Delattre|
|NULL| Pierre-Alexandre|               Fonta|
|NULL|              Jan|               Krepl|
|NULL|        Stanislav|          

In [77]:
from pyspark.sql.functions import concat_ws

In [78]:
cr_authors_ready = cr_author_tmp.withColumn("full_name", concat_ws(" ", "given", "family")).select("full_name", col("ORCID").alias("orcid"), "doi")

In [79]:
cr_authors_ready.show()

+--------------------+-----+--------------------+
|           full_name|orcid|                 doi|
+--------------------+-----+--------------------+
|Christopher T. Le...| NULL|10.1101/2020.05.2...|
|           Edsel Ing| NULL|10.1101/2020.05.2...|
|    Joseph D. Lykins| NULL|10.1101/2020.05.2...|
|    Matthew C. Hogan| NULL|10.1101/2020.05.2...|
|    Craig A. McKeown| NULL|10.1101/2020.05.2...|
|  Andrzej Grzybowski| NULL|10.1101/2020.05.2...|
|  Emmanuelle Logette| NULL|10.1101/2021.04.2...|
|     Charlotte Lorin| NULL|10.1101/2021.04.2...|
|     Cyrille Favreau| NULL|10.1101/2021.04.2...|
|     Eugenia Oshurko| NULL|10.1101/2021.04.2...|
|       Jay S. Coggan| NULL|10.1101/2021.04.2...|
| Francesco Casalegno| NULL|10.1101/2021.04.2...|
|Mohameth François Sy| NULL|10.1101/2021.04.2...|
|      Caitlin Monney| NULL|10.1101/2021.04.2...|
|     Marine Bertschy| NULL|10.1101/2021.04.2...|
|     Emilie Delattre| NULL|10.1101/2021.04.2...|
|Pierre-Alexandre ...| NULL|10.1101/2021.04.2...|


In [80]:
cr_authors_ready.printSchema()

root
 |-- full_name: string (nullable = false)
 |-- orcid: string (nullable = true)
 |-- doi: string (nullable = true)



In [81]:
cr_authors_ready.count()

353014

In [83]:
from pyspark.sql.functions import col, trim

In [93]:
cr_authors_ready_clean = cr_authors_ready.filter(col("full_name").isNotNull() & (trim(col("full_name")) != "")).filter(col("orcid").isNotNull() & (trim(col("orcid")) != ""))

In [94]:
cr_authors_ready_clean.count()

87108

In [95]:
cr_authors_ready_clean.show()

+--------------------+--------------------+--------------------+
|           full_name|               orcid|                 doi|
+--------------------+--------------------+--------------------+
|          Jo Yi Chow|https://orcid.org...|10.1101/2025.09.2...|
|Muttanagouda Giri...|https://orcid.org...|10.1101/2022.02.0...|
|   Basavaraj Vastrad|https://orcid.org...|10.1101/2022.02.0...|
| Rajeshwari Horakeri|https://orcid.org...|10.1101/2022.02.0...|
|Chanabasayya Vastrad|https://orcid.org...|10.1101/2022.02.0...|
| Abhijit Chakraborty|https://orcid.org...|10.1101/2020.05.2...|
|        Jiaying Chen|https://orcid.org...|10.1101/2020.05.2...|
|Amélie Desvars-La...|https://orcid.org...|10.1101/2020.05.2...|
|        Peter Klimek|https://orcid.org...|10.1101/2020.05.2...|
|  Erwin Flores Tames|https://orcid.org...|10.1101/2020.05.2...|
|        David Garcia|https://orcid.org...|10.1101/2020.05.2...|
| Leonhard Horstmeyer|https://orcid.org...|10.1101/2020.05.2...|
|     Michaela Kaleta|htt

In [96]:
cr_authors_ready_clean.printSchema()

root
 |-- full_name: string (nullable = false)
 |-- orcid: string (nullable = true)
 |-- doi: string (nullable = true)



In [97]:
df_authors.printSchema()

root
 |-- doi: string (nullable = true)
 |-- full_name: string (nullable = true)



In [98]:
from pyspark.sql.functions import lower, trim, col

In [99]:
df_joined = df_authors.alias("a").join(
    cr_authors_ready_clean.alias("b"),
    (
        lower(trim(col("a.full_name"))) ==
        lower(trim(col("b.full_name")))
    ),
    how="left"
).select(
    col("a.doi"),
    col("a.full_name"),
    col("b.orcid")
)

In [100]:
df_joined.show(20, False)

[Stage 87:=====================================================>(981 + 3) / 984]

+---------------------------+-------------------+-------------------------------------+
|doi                        |full_name          |orcid                                |
+---------------------------+-------------------+-------------------------------------+
|10.1101/2022.04.20.22274090|Rachel L Zapf      |NULL                                 |
|10.1101/2022.04.20.22274090|Paula L Kip        |NULL                                 |
|10.1101/2022.04.20.22274097|Meng Liu           |NULL                                 |
|10.1101/2022.04.21.22274025|Ashley David Otter |https://orcid.org/0000-0002-8317-9194|
|10.1101/2022.04.21.22274025|Ashley David Otter |https://orcid.org/0000-0002-8317-9194|
|10.1101/2022.04.21.22274025|Ashley David Otter |https://orcid.org/0000-0002-8317-9194|
|10.1101/2022.04.20.22274097|Jonathan M Berman  |NULL                                 |
|10.1101/2022.04.20.22274090|Christina Megli    |NULL                                 |
|10.1101/2022.04.21.22274025|Hea

In [101]:
df_final = df_raw_clean.join(df_joined, on="doi", how="inner")

In [102]:
df_final.printSchema()

root
 |-- doi: string (nullable = true)
 |-- category: string (nullable = true)
 |-- create_date: string (nullable = true)
 |-- type: string (nullable = true)
 |-- version: string (nullable = true)
 |-- title: string (nullable = true)
 |-- link: string (nullable = true)
 |-- site: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- orcid: string (nullable = true)



In [103]:
df_final.write.mode("overwrite").json("../data/output/df_final")

In [6]:
df.createOrReplaceTempView("df")

In [7]:
df_sql = spark.sql("""SELECT explode(collection) as item from df""")

In [8]:
df_sql.printSchema()

root
 |-- item: struct (nullable = true)
 |    |-- category: string (nullable = true)
 |    |-- license: string (nullable = true)
 |    |-- rel_abs: string (nullable = true)
 |    |-- rel_authors: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- author_inst: string (nullable = true)
 |    |    |    |-- author_name: string (nullable = true)
 |    |-- rel_date: string (nullable = true)
 |    |-- rel_doi: string (nullable = true)
 |    |-- rel_link: string (nullable = true)
 |    |-- rel_num_authors: long (nullable = true)
 |    |-- rel_site: string (nullable = true)
 |    |-- rel_title: string (nullable = true)
 |    |-- type: string (nullable = true)
 |    |-- version: string (nullable = true)



In [9]:
df_sql2 = spark.sql("""SELECT item.* FROM (SELECT explode(collection) as item from df)""")

In [10]:
df_sql2.printSchema()

root
 |-- category: string (nullable = true)
 |-- license: string (nullable = true)
 |-- rel_abs: string (nullable = true)
 |-- rel_authors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- author_inst: string (nullable = true)
 |    |    |-- author_name: string (nullable = true)
 |-- rel_date: string (nullable = true)
 |-- rel_doi: string (nullable = true)
 |-- rel_link: string (nullable = true)
 |-- rel_num_authors: long (nullable = true)
 |-- rel_site: string (nullable = true)
 |-- rel_title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- version: string (nullable = true)

